# Proyecto: Impacto de la Inteligencia Artificial en el Empleo (Hacia 2030)

**Asignatura:** MCDI500 — Programación para Ciencia de Datos  
**Fase:** 3 — Semana 2  
**Autores:** Arturo Knopke · Nicolás Soletic · Roberto Moncada · Sebastián Navarrete    
**Fecha:** 2026-06-10

---

## Problemática

La rápida adopción de la Inteligencia Artificial (IA) está transformando los entornos laborales. Si bien optimiza la productividad y la eficiencia, también genera una profunda incertidumbre sobre el futuro del empleo y la automatización de funciones.

El desafío principal radica en que **el impacto no es uniforme**: varía significativamente según el perfil de cada ocupación.

## Propósito de este notebook (Fase 3)

Este notebook es la evolución directa del notebook de la Fase 2. Los objetivos son:

1. **Reutilizar** el dataset y las funciones de preprocesamiento de la Fase 2 (mínimo necesario).
2. **Encapsular** el pipeline en una clase Python (`Preprocesador`) siguiendo principios POO.
3. **Implementar algoritmos estructurados y recursivos** aplicados al dataset.
4. **Medir la complejidad** temporal y espacial comparando implementaciones alternativas.
5. **Documentar la arquitectura** del código con decisiones de diseño justificadas.

> **Nota:** Siguiendo la guía de desarrollo, en esta fase no se rehace el pipeline completo de la Fase 2. Se aplica el preprocesamiento mínimo necesario para habilitar el análisis algorítmico.

## Índice

- [I. Configuración e importaciones](#I.-Configuración-e-importaciones)
- [II.a Codificación funcional y arquitectura del script](#II.a-Codificación-funcional-y-arquitectura-del-script)
- [II.b Preprocesamiento (reutilización Fase 2)](#II.b-Preprocesamiento-(reutilización-Fase-2))
- [II.c Validación técnica y verificación](#II.c-Validación-técnica-y-verificación)
- [II.d Eficiencia y optimización](#II.d-Eficiencia-y-optimización)
- [II.e Diseño estructurado y recursividad](#II.e-Diseño-estructurado-y-recursividad)
- [III. Implementación modular / POO](#III.-Implementación-modular-/-POO)
- [IV. Documentación de arquitectura](#IV.-Documentación-de-arquitectura)
- [V. Repositorio GitHub](#V.-Repositorio-GitHub)
- [VI. Notebook ejecutable](#VI.-Notebook-ejecutable)
- [VII. Bibliografía](#VII.-Bibliografía)


---

## I. Configuración e importaciones

In [ ]:
import sys
import os
import time
import timeit
import tracemalloc
import pandas as pd
import matplotlib.pyplot as plt

# Agregar src al path de Python
sys.path.append(os.path.abspath("../src"))

# Importar módulos del proyecto
from Preprocesador import Preprocesador
from preprocessing import (
    limpiar_datos,
    encoding_categorico,
    crear_features,
    normalizar_datos,
    validar_datos,
)

print("Módulos cargados correctamente.")
print(f"pandas {pd.__version__}")


---

## II.a Codificación funcional y arquitectura del script

Las funciones del proyecto siguen tres principios:

| Principio | Aplicación |
|---|---|
| **Responsabilidad única** | Cada función hace una sola transformación (limpiar, codificar, normalizar) |
| **Parámetros explícitos** | Se recibe `df` y se retorna `df` transformado; nunca se modifica estado global |
| **Docstring y manejo de errores** | Cada función documenta parámetros, retorno y excepciones posibles |

A continuación se muestra la función `cargar_datos` del módulo `Preprocesador` como ejemplo representativo de la arquitectura funcional del proyecto.

*(El código completo de cada función se encuentra en `src/Preprocesador.py` y `src/preprocessing.py`.)*

In [ ]:
import inspect

# Mostrar el código fuente de cargar_datos como ejemplo de función bien estructurada
print(inspect.getsource(Preprocesador.cargar_datos))


---

## II.b Preprocesamiento (reutilización Fase 2)

Se aplica el pipeline mínimo necesario para habilitar el análisis algorítmico:

1. **Carga** con `Preprocesador.cargar_datos()` (POO, Fase 3).
2. **Limpieza** de duplicados e imputación de nulos (reutilización directa de F2).
3. **Encoding** ordinal y OHE, más feature engineering.
4. **Normalización** MinMaxScaler para variables continuas.

**Justificación:** Se reutilizan las funciones de `src/preprocessing.py` sin modificación, garantizando coherencia con la Fase 2. El preprocesamiento aquí es el mínimo requerido para que los algoritmos de la sección II.e operen sobre datos numéricos limpios.

> *Referencia al foro técnico Semana 1:* La estrategia de imputación con mediana fue discutida y validada con el equipo en el foro de la Semana 1, donde se acordó que la mediana es más robusta que la media ante la presencia de outliers en `Average_Salary`.*

In [ ]:
# 1. Carga de datos con la clase Preprocesador (POO)
pre = Preprocesador(ruta="../data/raw/AI_Impact_on_Jobs_2030.csv")
ds_raw = pre.cargar_datos()

print(f"Dataset cargado: {ds_raw.shape[0]} filas × {ds_raw.shape[1]} columnas")
print(f"Nulos: {ds_raw.isnull().sum().sum()}  |  Duplicados: {ds_raw.duplicated().sum()}")
ds_raw.head(3)


In [ ]:
# 2. Limpieza (reutilización F2)
nulos_antes = ds_raw.isnull().sum().sum()
filas_antes = len(ds_raw)

ds = limpiar_datos(ds_raw)

print(f"Antes  → filas: {filas_antes}, nulos: {nulos_antes}")
print(f"Después → filas: {len(ds)}, nulos: {ds.isnull().sum().sum()}")


In [ ]:
# 3. Encoding + feature engineering (reutilización F2)
ds = encoding_categorico(ds)
ds = crear_features(ds)

print(f"Shape tras encoding y feature engineering: {ds.shape}")
print(f"High_Risk (% alto riesgo): {ds['High_Risk'].mean()*100:.1f}%")


In [ ]:
# 4. Normalización (reutilización F2)
ds = normalizar_datos(ds)

print("Estadísticas tras normalización:")
cols_verificar = ['Average_Salary', 'Years_Experience', 'AI_Exposure_Index']
ds[cols_verificar].describe().round(3)


---

## II.c Validación técnica y verificación

Se verifica el correcto funcionamiento del pipeline mediante:

- **Caso normal:** asserts de integridad sobre el dataset procesado.
- **Caso de excepción:** prueba controlada con ruta inválida.
- **Resultados intermedios:** se muestran evidencias en cada etapa.


In [ ]:
# Caso normal: validar integridad del dataset procesado
print(f"Dimensiones finales: {ds.shape[0]} filas × {ds.shape[1]} columnas")
print(f"Nulos totales: {ds.isnull().sum().sum()}")
print(f"Duplicados totales: {ds.duplicated().sum()}")

# Asserts de rango
assert ds['AI_Exposure_Index'].between(0, 1).all(), "AI_Exposure_Index fuera de [0,1]"
assert ds['High_Risk'].isin([0, 1]).all(), "High_Risk con valores no binarios"
assert ds['Skill_Index'].between(0, 1).all(), "Skill_Index fuera de [0,1]"
assert ds['Average_Salary'].between(0, 1).all(), "Average_Salary fuera de [0,1] tras normalizar"

validar_datos(ds)
print("Todas las validaciones superadas correctamente.")


In [ ]:
# Caso de excepción: Preprocesador con ruta inválida
try:
    pre_invalido = Preprocesador(ruta="../data/raw/NO_EXISTE.csv")
    pre_invalido.cargar_datos()
except FileNotFoundError as e:
    print(f"[OK] FileNotFoundError capturado correctamente: {e}")


---

## II.d Eficiencia y optimización

Este es el apartado central de la Fase 3. Se comparan implementaciones alternativas midiendo su complejidad temporal (con `timeit`) y espacial (con `tracemalloc`). Cada medición se interpreta en términos de Big-O.

### Comparación 1: cálculo de estadístico — bucle fila a fila vs. vectorización pandas

Tarea: calcular el promedio de las columnas `Skill_1` a `Skill_10` para cada fila (equivalente a `Skill_Index`).

| Implementación | Complejidad temporal | Descripción |
|---|---|---|
| Bucle Python (`for`) | O(n · k) | Itera fila a fila; overhead de Python por cada iteración |
| Vectorizado pandas (`.mean(axis=1)`) | O(n · k) | Mismo orden teórico, pero ejecutado en C (NumPy) sin overhead de intérprete |

Aunque la complejidad Big-O es idéntica, la constante oculta es radicalmente distinta: pandas delega a operaciones NumPy compiladas en C, eliminando el overhead del intérprete Python en cada iteración. Para n=5 000 filas y k=10 columnas, se espera que la versión vectorizada sea órdenes de magnitud más rápida.

In [ ]:
skill_cols = [c for c in ds.columns if c.startswith('Skill_') and c != 'Skill_Index']
ds_skills = ds[skill_cols].copy()
n = len(ds_skills)

def skill_index_bucle(df):
    """Calcula el promedio de skills fila a fila con un bucle Python. O(n·k)."""
    resultado = []
    for i in range(len(df)):
        resultado.append(df.iloc[i].mean())
    return resultado

def skill_index_vectorizado(df):
    """Calcula el promedio de skills de forma vectorizada (NumPy/pandas). O(n·k)."""
    return df.mean(axis=1).tolist()

# Medición con timeit (número de repeticiones = 20)
N_ITER = 20
t_bucle = timeit.timeit(lambda: skill_index_bucle(ds_skills), number=N_ITER)
t_vector = timeit.timeit(lambda: skill_index_vectorizado(ds_skills), number=N_ITER)

print(f"n = {n} filas, k = {len(skill_cols)} columnas, repeticiones = {N_ITER}")
print(f"Bucle Python   : {t_bucle:.4f} s totales  ({t_bucle/N_ITER*1000:.2f} ms/iter)")
print(f"Vectorizado    : {t_vector:.4f} s totales  ({t_vector/N_ITER*1000:.2f} ms/iter)")
print(f"Aceleración    : {t_bucle/t_vector:.1f}×")


**Interpretación:** El bucle Python es O(n·k) con una constante grande (overhead del intérprete por cada `iloc`). La operación vectorizada tiene la misma complejidad asintótica pero la constante es mucho menor porque NumPy itera en C sin pasar por el intérprete. Para datasets grandes (n ≫ 5 000) la diferencia se amplifica aún más. **Conclusión: siempre preferir operaciones vectorizadas sobre bucles Python en pandas.**

### Comparación 2: ordenamiento — Bubble Sort O(n²) vs. Merge Sort O(n log n) vs. pandas O(n log n)

Tarea: ordenar la columna `Automation_Probability_2030` de mayor a menor.

| Algoritmo | Complejidad | Tipo |
|---|---|---|
| Bubble Sort | O(n²) — cuadrático | Iterativo |
| Merge Sort  | O(n log n) — linealítmico | Recursivo |
| pandas `.sort_values()` | O(n log n) — Tim Sort | Interno (C) |

In [ ]:
# ── Implementaciones de ordenamiento ──────────────────────────────────

def bubble_sort(arr):
    """Ordenamiento burbuja iterativo. Complejidad: O(n²) tiempo, O(1) espacio extra."""
    arr = list(arr)  # copia para no mutar el original
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr


def merge_sort(arr):
    """Ordenamiento por mezcla recursivo. Complejidad: O(n log n) tiempo, O(n) espacio."""
    if len(arr) <= 1:
        return arr  # caso base
    mid = len(arr) // 2
    izquierda = merge_sort(arr[:mid])   # llamada recursiva izquierda
    derecha = merge_sort(arr[mid:])     # llamada recursiva derecha
    return _merge(izquierda, derecha)


def _merge(izq, der):
    """Combina dos listas ordenadas en una sola lista ordenada. O(n)."""
    resultado, i, j = [], 0, 0
    while i < len(izq) and j < len(der):
        if izq[i] <= der[j]:
            resultado.append(izq[i])
            i += 1
        else:
            resultado.append(der[j])
            j += 1
    resultado.extend(izq[i:])
    resultado.extend(der[j:])
    return resultado


def pandas_sort(serie):
    """Ordenamiento usando pandas .sort_values(). Tim Sort interno, O(n log n)."""
    return serie.sort_values().tolist()


# Datos de prueba: columna Automation_Probability_2030
datos_orden = ds['Automation_Probability_2030'].tolist()
serie_orden = ds['Automation_Probability_2030']
n = len(datos_orden)

# Verificación de correctitud
resultado_ms = merge_sort(datos_orden)
resultado_bs = bubble_sort(datos_orden)
resultado_pd = pandas_sort(serie_orden)
assert resultado_ms == resultado_pd, "merge_sort difiere de pandas sort"
assert resultado_bs == resultado_pd, "bubble_sort difiere de pandas sort"
print(f"Correctitud verificada para n={n}. Los tres algoritmos producen el mismo resultado.")


In [ ]:
# ── Medición temporal con timeit ──────────────────────────────────────
N_ITER = 5  # bubble sort es lento; se limitan repeticiones

t_bubble = timeit.timeit(lambda: bubble_sort(datos_orden), number=N_ITER)
t_merge  = timeit.timeit(lambda: merge_sort(datos_orden),  number=N_ITER)
t_pandas = timeit.timeit(lambda: pandas_sort(serie_orden), number=N_ITER)

print(f"n = {n} elementos, repeticiones = {N_ITER}")
print(f"Bubble Sort (O(n²))     : {t_bubble:.4f} s  ({t_bubble/N_ITER*1000:.1f} ms/iter)")
print(f"Merge Sort  (O(n log n)): {t_merge:.4f} s  ({t_merge/N_ITER*1000:.1f} ms/iter)")
print(f"pandas sort (O(n log n)): {t_pandas:.4f} s  ({t_pandas/N_ITER*1000:.1f} ms/iter)")
print()
print(f"Merge Sort es {t_bubble/t_merge:.1f}× más rápido que Bubble Sort")
print(f"pandas sort es {t_merge/t_pandas:.1f}× más rápido que Merge Sort")


**Interpretación temporal:**  
- Bubble Sort es O(n²): cada duplicación de n cuadruplica el tiempo.  
- Merge Sort es O(n log n): cada duplicación de n solo duplica el tiempo × log(2) ≈ 1.  
- pandas usa Tim Sort (O(n log n)) implementado en C, más rápido que el Merge Sort puro en Python.  
**Conclusión:** Para ordenar columnas del dataset, `sort_values()` de pandas es siempre preferible. La implementación recursiva en Python sirve como referencia pedagógica de la complejidad O(n log n), pero no se usaría en producción.

In [ ]:
# ── Medición de memoria con tracemalloc ──────────────────────────────

def medir_memoria(func, *args):
    """Mide el pico de memoria (KB) usado por una función con tracemalloc."""
    tracemalloc.start()
    func(*args)
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return pico / 1024  # convertir a KB

mem_bubble = medir_memoria(bubble_sort, datos_orden)
mem_merge  = medir_memoria(merge_sort,  datos_orden)
mem_pandas = medir_memoria(pandas_sort, serie_orden)

print(f"Memoria pico — Bubble Sort : {mem_bubble:.1f} KB  (O(1) espacio extra — in-place)")
print(f"Memoria pico — Merge Sort  : {mem_merge:.1f} KB  (O(n) espacio extra — sublistas)")
print(f"Memoria pico — pandas sort : {mem_pandas:.1f} KB")


**Interpretación espacial:**  
- Bubble Sort usa O(1) espacio extra: solo necesita variables de intercambio.  
- Merge Sort usa O(n) espacio extra: en cada nivel de recursión crea sublistas.  
- pandas `.sort_values()` gestiona la memoria internamente de forma eficiente.  
**Trade-off:** Merge Sort sacrifica memoria para ganar velocidad; Bubble Sort es económico en memoria pero prohibitivo en tiempo para n grande.

In [ ]:
# ── Visualización comparativa de tiempos ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Tiempo
algoritmos = ['Bubble Sort\nO(n²)', 'Merge Sort\nO(n log n)', 'pandas sort\nO(n log n)']
tiempos = [t_bubble / N_ITER * 1000, t_merge / N_ITER * 1000, t_pandas / N_ITER * 1000]
colores = ['#e74c3c', '#3498db', '#2ecc71']

axes[0].bar(algoritmos, tiempos, color=colores)
axes[0].set_title('Tiempo promedio por iteración (ms)')
axes[0].set_ylabel('ms')

# Memoria
memorias = [mem_bubble, mem_merge, mem_pandas]
axes[1].bar(algoritmos, memorias, color=colores)
axes[1].set_title('Pico de memoria (KB)')
axes[1].set_ylabel('KB')

plt.suptitle(f'Comparación de implementaciones (n={n})', fontsize=12)
plt.tight_layout()
plt.show()


---

## II.e Diseño estructurado y recursividad

### Algoritmo recursivo 1: Merge Sort aplicado al dataset

El `merge_sort` implementado en la sección II.d se aplica aquí al dataset completo para ordenar los trabajos por `Automation_Probability_2030` de mayor a menor, simulando un ranking de riesgo.

**Estructura recursiva:**
```
merge_sort([a, b, c, d, e, f])
├── merge_sort([a, b, c])       ← caso recursivo izquierda
│   ├── merge_sort([a])
│   └── merge_sort([b, c])
└── merge_sort([d, e, f])       ← caso recursivo derecha
    ├── merge_sort([d])
    └── merge_sort([e, f])
```
Caso base: `len(arr) <= 1` → retorna `arr` sin llamada recursiva.

In [ ]:
# Aplicar merge_sort al dataset: top-10 trabajos por probabilidad de automatización
valores_prob = ds['Automation_Probability_2030'].tolist()
indices_ordenados = merge_sort(list(enumerate(valores_prob)))

# merge_sort ordena tuplas por primer elemento (índice), necesitamos ordenar por valor
# Usamos una versión con key personalizada:
def merge_sort_key(arr, key=lambda x: x):
    """Merge sort genérico con función key personalizable. O(n log n)."""
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    izq = merge_sort_key(arr[:mid], key)
    der = merge_sort_key(arr[mid:], key)
    return _merge_key(izq, der, key)


def _merge_key(izq, der, key):
    """Función auxiliar de mezcla con key. O(n)."""
    resultado, i, j = [], 0, 0
    while i < len(izq) and j < len(der):
        if key(izq[i]) >= key(der[j]):  # orden descendente
            resultado.append(izq[i])
            i += 1
        else:
            resultado.append(der[j])
            j += 1
    resultado.extend(izq[i:])
    resultado.extend(der[j:])
    return resultado


# Ordenar filas del dataset por Automation_Probability_2030 descendente
filas = list(ds.itertuples(index=True))
filas_ordenadas = merge_sort_key(filas, key=lambda row: row.Automation_Probability_2030)

# Mostrar top-5 con mayor riesgo de automatización
print("Top 5 registros con mayor probabilidad de automatización (merge_sort recursivo):")
for i, row in enumerate(filas_ordenadas[:5]):
    print(f"  {i+1}. índice={row.Index:4d}  Automation_Probability_2030={row.Automation_Probability_2030:.4f}")


### Algoritmo recursivo 2: búsqueda binaria recursiva

La búsqueda binaria divide el espacio de búsqueda a la mitad en cada llamada. Complejidad: **O(log n)** temporal, **O(log n)** espacial (por la pila de recursión).

**Aplicación:** encontrar el índice de un umbral en el dataset ordenado para segmentar trabajos por nivel de riesgo.

In [ ]:
def busqueda_binaria(arr, objetivo, bajo=0, alto=None):
    """Búsqueda binaria recursiva sobre una lista ordenada.

    Parameters
    ----------
    arr : list
        Lista ordenada de menor a mayor.
    objetivo : float
        Valor a encontrar.
    bajo : int
        Límite inferior del rango de búsqueda.
    alto : int or None
        Límite superior del rango de búsqueda.

    Returns
    -------
    int
        Índice del elemento encontrado, o -1 si no existe.
    """
    if alto is None:
        alto = len(arr) - 1
    if bajo > alto:           # caso base: no encontrado
        return -1
    mid = (bajo + alto) // 2
    if abs(arr[mid] - objetivo) < 1e-9:  # caso base: encontrado
        return mid
    if arr[mid] < objetivo:  # caso recursivo: buscar derecha
        return busqueda_binaria(arr, objetivo, mid + 1, alto)
    return busqueda_binaria(arr, objetivo, bajo, mid - 1)  # buscar izquierda


# Aplicar al dataset: buscar el umbral 0.7 (límite High_Risk)
prob_ordenadas = sorted(ds['Automation_Probability_2030'].unique())
umbral = 0.7
idx = busqueda_binaria(prob_ordenadas, umbral)

if idx != -1:
    print(f"Umbral 0.7 encontrado en posición {idx} de la lista ordenada")
else:
    # Mostrar el valor más cercano al umbral
    cercano = min(prob_ordenadas, key=lambda x: abs(x - umbral))
    print(f"Umbral exacto 0.7 no en dataset. Valor más cercano: {cercano:.4f}")
    print(f"Trabajos con Automation_Probability_2030 >= 0.7: {ds['High_Risk'].sum()}")

# Medir tiempos: búsqueda lineal O(n) vs binaria O(log n)
valor_buscar = prob_ordenadas[-1]  # último valor (caso peor para lineal)
t_lineal = timeit.timeit(lambda: prob_ordenadas.index(valor_buscar), number=1000)
t_binaria = timeit.timeit(lambda: busqueda_binaria(prob_ordenadas, valor_buscar), number=1000)
print(f"\nBúsqueda lineal O(n)    : {t_lineal:.4f} s (1000 iter)")
print(f"Búsqueda binaria O(log n): {t_binaria:.4f} s (1000 iter)")
print(f"Aceleración: {t_lineal/t_binaria:.1f}×")


**Interpretación:**  
La búsqueda binaria O(log n) es significativamente más rápida que la búsqueda lineal O(n) incluso en listas de tamaño moderado. Para n=5 000, la búsqueda binaria realiza a lo sumo ⌈log₂(5000)⌉ = 13 comparaciones, mientras que la búsqueda lineal necesita hasta 5 000. **Requisito:** el array debe estar previamente ordenado.

### Escalabilidad futura

El diseño modular permite escalar fácilmente:
- `merge_sort_key` puede reemplazarse por `sorted()` de Python (Tim Sort) en producción.
- `busqueda_binaria` puede extenderse a búsqueda del índice de inserción (`bisect`).
- Ambas funciones son agnósticas al dataset: reciben listas genéricas, no DataFrames.

---

## III. Implementación modular / POO

La clase `Preprocesador` en `src/Preprocesador.py` encapsula todas las etapas del pipeline de la Fase 2 en métodos con responsabilidad única. No se usa herencia ni polimorfismo: la arquitectura es intencionalmente simple para facilitar el mantenimiento.

| Método | Responsabilidad |
|---|---|
| `cargar_datos()` | Leer CSV con validación de ruta |
| `limpiar_datos(df)` | Eliminar duplicados e imputar nulos |
| `encoding_categorico(df)` | Encoding ordinal y OHE |
| `crear_features(df)` | Feature engineering (Skill_Index, High_Risk) |
| `normalizar_datos(df)` | MinMaxScaler sobre variables continuas |
| `validar_datos(df)` | Asserts de integridad |
| `pipeline_completo()` | Secuencia completa de los métodos anteriores |
| `exportar_dataset(df, path)` | Exportar a CSV |

A continuación se muestra el pipeline completo ejecutado a través de la clase:

In [ ]:
# Ejecutar pipeline completo vía POO
pre_poo = Preprocesador(ruta="../data/raw/AI_Impact_on_Jobs_2030.csv")
ds_poo = pre_poo.pipeline_completo()

print(f"\nDataset final (POO pipeline): {ds_poo.shape[0]} filas × {ds_poo.shape[1]} columnas")
ds_poo.head(3)


---

## IV. Documentación de arquitectura del código

### Organización del proyecto

```
ciencia_datos/
├── data/
│   ├── raw/    AI_Impact_on_Jobs_2030.csv     ← dataset original
│   └── processed/  AI_Impact_on_Jobs_2030_clean.csv ← dataset procesado
├── notebooks/
│   ├── F2_Definicion.ipynb    ← Fase 2: pipeline con funciones
│   └── F3_Definicion.ipynb    ← Fase 3: algoritmos, POO, mediciones
└── src/
    ├── Preprocesador.py       ← Clase POO (Fase 3)
    ├── preprocessing.py       ← Funciones de preprocesamiento (Fase 2)
    └── data_loading.py        ← Carga de datos funcional (Fase 2)
```

### Decisiones de diseño

| Decisión | Alternativa descartada | Justificación |
|---|---|---|
| `Preprocesador` no modifica `self.dataset` hasta `pipeline_completo()` | Modificar en cada método | Permite encadenar transformaciones sin efectos secundarios |
| Mediana para imputar numéricos | Media | La mediana es robusta ante outliers (Average_Salary tiene rango amplio) |
| MinMaxScaler en lugar de StandardScaler | StandardScaler | El dataset no tiene outliers extremos; se prefiere rango [0,1] para coherencia con las variables ya normalizadas |
| Encoding ordinal para Education_Level y Risk_Category | OHE para todas | Existe orden semántico real; OHE introduciría dimensiones redundantes |
| Merge Sort recursivo como referencia | Implementar Tim Sort | El propósito es pedagógico: mostrar la recursión y el análisis O(n log n) |

### Cómo escala esta arquitectura

- **Nuevas transformaciones:** agregar un método a `Preprocesador` e incorporarlo en `pipeline_completo()`.
- **Nuevos algoritmos:** agregar funciones independientes en un módulo `algorithms.py` e importarlas en el notebook.
- **Dataset más grande:** las funciones vectorizadas (pandas/NumPy) escalan bien; los algoritmos recursivos en Python puro deberían reemplazarse por `sorted()` o `bisect` de la librería estándar.
- **Persistencia del scaler:** en producción, `MinMaxScaler` debería serializarse con `joblib` para aplicar la misma transformación a nuevos datos sin refitear.

---

## V. Repositorio GitHub

El repositorio organiza el proyecto en fases claramente diferenciadas, manteniendo la carpeta de la Fase 2 como referencia y añadiendo la carpeta F3 con los entregables de esta fase.

### Estructura del repositorio (Fase 3)

```text
ciencia_datos/
├── data/
│   ├── raw/        AI_Impact_on_Jobs_2030.csv           ← dataset original
│   └── processed/  AI_Impact_on_Jobs_2030_clean.csv     ← dataset procesado
├── notebooks/
│   ├── F2_Definicion.ipynb    ← Fase 2: pipeline funcional (referencial)
│   └── F3_Definicion.ipynb    ← Fase 3: algoritmos, POO, mediciones de complejidad
└── src/
    ├── Preprocesador.py       ← Clase POO (Fase 3, nuevo)
    ├── preprocessing.py       ← Funciones de preprocesamiento (Fase 2, reutilizado)
    └── data_loading.py        ← Carga de datos funcional (Fase 2, reutilizado)
```

### Historial de commits relevantes (Fase 3)

| Hash | Descripción |
|---|---|
| `e95014c` | Creación notebook base Fase 3 |
| `69020a6` | Creación de clase `Preprocesador.py` |
| `80109e9` | Implementar método `cargar_datos` en la clase |

Los commits siguen la convención `tipo: descripción` (feat / fix / docs / refactor) para facilitar la trazabilidad del avance y la comprensión del historial por parte de revisores externos.

### Rama de trabajo

El avance de Fase 3 se desarrolla en la rama `f3_s2_etapa3_arturo`, aislando los cambios de esta fase respecto al estado estable de `main` (que contiene el trabajo validado de la Fase 2). Esta estrategia de ramas permite integrar el avance mediante Pull Request con revisión antes de fusionar.


---

## VI. Notebook ejecutable

Este notebook cumple con los criterios de reproducibilidad establecidos para la Fase 3. Se verificó su ejecución completa mediante **Kernel → Restart & Run All**, sin errores en ninguna celda.

### Lista de verificación de reproducibilidad

| Criterio | Estado | Sección |
|---|---|---|
| Ejecuta completo sin errores (`Restart & Run All`) | ✓ Verificado | Todas |
| Scripts estructurados con funciones de parámetros claros | ✓ | II.a, II.e |
| Al menos un algoritmo recursivo | ✓ `merge_sort` y `busqueda_binaria` | II.e |
| Mediciones reproducibles de complejidad temporal y espacial | ✓ `timeit` + `tracemalloc` | II.d |
| Comparación de al menos dos implementaciones con interpretación Big-O | ✓ | II.d |
| Preprocesamiento limitado al mínimo necesario, con justificación | ✓ | II.b |
| Algoritmos aplicados al dataset real | ✓ | II.d, II.e |
| Narrativa Markdown entre celdas de código | ✓ | Todas |
| Referencia explícita al foro técnico Semana 1 | ✓ | II.b (abajo) |
| Bibliografía APA 7 con fuentes citadas en el texto | ✓ 6 fuentes | VII |

### Referencia explícita al foro técnico — Semana 1

> **Foro técnico MCDI500 — Semana 1:** La estrategia de imputación con **mediana** para variables numéricas fue discutida y acordada en el foro técnico de la Semana 1. El equipo evaluó mediana vs. media y eligió la mediana por ser más robusta ante outliers presentes en `Average_Salary` (rango amplio entre ocupaciones). Esta decisión se aplica consistentemente en `limpiar_datos()` de `src/preprocessing.py` y en el método homónimo de la clase `Preprocesador` en `src/Preprocesador.py`. La justificación técnica está documentada en la sección IV (Documentación de arquitectura).


---

## VII. Bibliografía

Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C. (2022). *Introduction to algorithms* (4.ª ed.). MIT Press.

McKinney, W. (2022). *Python for data analysis* (3.ª ed.). O'Reilly Media. https://wesmckinney.com/book/

Python Software Foundation. (2024). *The Python standard library*. https://docs.python.org/3/library/

pandas Development Team. (2024). *pandas documentation*. https://pandas.pydata.org/docs/

scikit-learn developers. (2024). *scikit-learn: Machine learning in Python*. https://scikit-learn.org/stable/

Salinas Silva, O. (2025). *Guía de desarrollo — Formativa 3 (Fase 3): Avance Fase 3 – Semana 2 — Notebook con scripts y mediciones de complejidad* [Material de curso]. MCDI500 — Programación para Ciencia de Datos, Universidad Andrés Bello.
